In [62]:
# imports
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import geopandas as gpd

In [63]:
data = Path.cwd().parent.joinpath('raw_data', 'rivers', 'rivers.shp')
df = gpd.read_file(data)
df

,OBJECTID,Name,Feature,State,Region,Miles,Shape__Len,geometry
0,1,None,Stream,AK,19,2.41,0.072752,"LINESTRING (-142.15419 68.5338, -142.15474 68...."
1,2,Bear Creek,Stream,AK,19,24.85,0.525073,"LINESTRING (-159.55596 63.89679, -159.5563 63...."
2,3,None,Stream,AK,19,5.67,0.154646,"LINESTRING (-160.40021 63.51687, -160.40115 63..."
3,4,None,Stream,AK,19,7.37,0.158399,"LINESTRING (-158.07714 56.6516, -158.07748 56...."
4,5,Fortress Creek,Stream,AK,19,16.18,0.417355,"LINESTRING (-153.03977 68.47539, -153.03851 68..."
...,...,...,...,...,...,...,...,...
112963,112964,Ouzel Creek,Stream,WY,17,10.21,0.161982,"LINESTRING (-110.88646 44.34214, -110.88684 44..."
112964,112965,Stone Cabin Creek,Stream Intermittent,WY,10,1.76,0.027754,"LINESTRING (-107.35869 42.82882, -107.35791 42..."
112965,112966,North Fork Owl Creek,Stream,WY,10,1.67,0.033351,"LINESTRING (-108.88976 43.69674, -108.8851 43...."
112966,112967,Pinto Creek,Stream,WY,10,2.01,0.036111,"LINESTRING (-105.70748 42.03634, -105.70175 42..."


In [64]:
df["Feature"].value_counts()

Feature
Stream                   67813
Stream Intermittent      30679
Artificial Path          12334
Canal                     1930
Intracoastal Waterway      149
Aqueduct                    63
Name: count, dtype: int64

**Notes**:
- Stream intermittent: seasonal stream, can't support nuclear cooling => drop
- Aqueduct: water transport infrastructure
- Intracoastal waterway: ...

In [65]:
df = df[df["Feature"] != "Stream Intermittent"]
print(df.shape)
df = df.drop(columns = ["Name", "State", "Region", "Shape__Len"])
df.columns = ["id", "type", "miles", "geometry"]
df = df.to_crs(epsg = 5070)
df

(82289, 8)


,id,type,miles,geometry
0,1,Stream,2.41,"LINESTRING (-2369332.198 5440129.53, -2369360...."
1,2,Stream,24.85,"LINESTRING (-3406564.714 5618272.951, -3406670..."
2,3,Stream,5.67,"LINESTRING (-3467358.4 5620581.668, -3467387.1..."
3,4,Stream,7.37,"LINESTRING (-3779306.83 4990540.942, -3779323...."
4,5,Stream,16.18,"LINESTRING (-2869669.889 5736231.632, -2869445..."
...,...,...,...,...
112961,112962,Artificial Path,11.10,"LINESTRING (-890127.695 2157187.988, -890155.6..."
112962,112963,Stream,4.37,"LINESTRING (-1078546.296 2086600.439, -1078625..."
112963,112964,Stream,10.21,"LINESTRING (-1178847.923 2464980.379, -1178871..."
112965,112966,Stream,1.67,"LINESTRING (-1031509.143 2370553.239, -1031144..."


In [66]:
county = Path.cwd().parent.joinpath('raw_data', 'county_boundaries_2025', 'tl_2025_us_county.shp')

df_county = gpd.read_file(county)
df_county = df_county[["GEOID", "NAMELSAD", "geometry"]]
df_county.columns = ["geo_id", "county_name", "geometry"]
df_county = df_county.to_crs(epsg = 5070)
df_county

,geo_id,county_name,geometry
0,40075,Kiowa County,"POLYGON ((-267035.259 1343980.561, -266490.502..."
1,46079,Lake County,"POLYGON ((-71100.31 2327435.998, -71100.293 23..."
2,37033,Caswell County,"POLYGON ((1489331.21 1620697.325, 1489336.836 ..."
3,48377,Presidio County,"POLYGON ((-857749.848 879396.771, -857744.209 ..."
4,39057,Greene County,"POLYGON ((1008169.795 1915287.022, 1008161.905..."
...,...,...,...
3230,53065,Stevens County,"POLYGON ((-1634783.911 2949984.436, -1634792.5..."
3231,19177,Van Buren County,"POLYGON ((319198.812 1994179.49, 319368.793 19..."
3232,31073,Gosper County,"POLYGON ((-333884.885 1963990.326, -333876.567..."
3233,28095,Monroe County,"POLYGON ((712737.052 1232057.495, 712736.922 1..."


In [67]:
rivers_combined = df.geometry.union_all()
df_county["centroid"] = df_county.centroid
df_county["distance_to_rivers_km"] = df_county["centroid"].apply(
    lambda pt: pt.distance(rivers_combined) / 1000
)
df_county.head()

,geo_id,county_name,geometry,centroid,distance_to_rivers_km
0,40075,Kiowa County,"POLYGON ((-267035.259 1343980.561, -266490.502...",POINT (-270070.971 1321674.415),6.959536
1,46079,Lake County,"POLYGON ((-71100.31 2327435.998, -71100.293 23...",POINT (-90223.338 2337302.331),3.555520
2,37033,Caswell County,"POLYGON ((1489331.21 1620697.325, 1489336.836 ...",POINT (1473665.516 1612333.443),0.560790
3,48377,Presidio County,"POLYGON ((-857749.848 879396.771, -857744.209 ...",POINT (-793168.208 803695.173),10.483502
4,39057,Greene County,"POLYGON ((1008169.795 1915287.022, 1008161.905...",POINT (1026417.426 1917896.79),3.454858


In [68]:
df_join = df.sjoin(df_county, how = "left", predicate = "intersects")

rivers_by_county = df_join.groupby(["geo_id", "county_name"]).agg(
    rivers_count = ("id", "count"),
    total_rivers_mile = ("miles", "sum")
).reset_index()

rivers_by_county

,geo_id,county_name,rivers_count,total_rivers_mile
0,01001,Autauga County,26,307.86
1,01003,Baldwin County,42,749.50
2,01005,Barbour County,34,457.11
3,01007,Bibb County,17,489.94
4,01009,Blount County,17,446.91
...,...,...,...,...
3170,72145,Vega Baja Municipio,4,29.42
3171,72147,Vieques Municipio,2,0.22
3172,72149,Villalba Municipio,8,22.24
3173,72151,Yabucoa Municipio,2,14.69


In [69]:
df_county.dtypes

geo_id                     object
county_name                object
geometry                 geometry
centroid                 geometry
distance_to_rivers_km     float64
dtype: object

In [70]:
rivers_by_county.dtypes

geo_id                object
county_name           object
rivers_count           int64
total_rivers_mile    float64
dtype: object

In [72]:
rivers_by_county = df_county.merge(rivers_by_county.drop(columns = "county_name"), how = "left", on = "geo_id")
rivers_by_county = rivers_by_county.drop(columns = ["geometry", "centroid"])
rivers_by_county

,geo_id,county_name,distance_to_rivers_km,rivers_count,total_rivers_mile
0,40075,Kiowa County,6.959536,27.0,688.07
1,46079,Lake County,3.555520,11.0,47.64
2,37033,Caswell County,0.560790,19.0,178.23
3,48377,Presidio County,10.483502,11.0,726.40
4,39057,Greene County,3.454858,13.0,276.85
...,...,...,...,...,...
3230,53065,Stevens County,0.027165,81.0,798.02
3231,19177,Van Buren County,1.819152,11.0,321.91
3232,31073,Gosper County,11.068131,7.0,97.63
3233,28095,Monroe County,3.045655,33.0,319.31


In [73]:
rivers_by_county.to_csv("../processed_data/rivers_by_county.csv")